# 🎭 Talking Avatar Server — Google Colab (T4)

Notebook siap-pakai untuk menjalankan endpoint `POST /avatar` (talk/idle) untuk fitur Commentary.

**Langkah:**
1. Menu **Runtime → Change runtime type → Hardware accelerator = T4 GPU**, lalu Save.
2. **Runtime → Run all** (atau jalankan cell satu per satu dari atas).
3. Tunggu sampai cell terakhir mencetak URL publik `https://xxxx.trycloudflare.com`.
4. Salin URL itu ke app (setting `avatarColabUrl`, atau pakai `xttsColabUrl` yang sama).

> Catatan T4: resolusi dikunci ≤512 & fp16. Endpoint avatar butuh menitan — pastikan timeout di app besar. Colab free bisa putus sesi; untuk produksi rutin pertimbangkan Colab Pro.

In [ ]:
# 1) Cek GPU (harus tampil Tesla T4)
!nvidia-smi

In [ ]:
# 2) Clone repo (branch feat/talking-avatar)
%cd /content
![ -d yt-short ] || git clone -b feat/talking-avatar https://github.com/dionisius95/yt-short.git
%cd /content/yt-short/colab

In [ ]:
# 3) Install engine avatar (SadTalker + LivePortrait). Jalankan sekali per sesi (~beberapa menit).
!bash setup_colab.sh

In [ ]:
# 4) Set environment engine
import os
os.environ['SADTALKER_DIR']    = '/content/SadTalker'
os.environ['LIVEPORTRAIT_DIR'] = '/content/LivePortrait'
os.environ['IDLE_DRIVING']     = '/content/assets/idle_driving.mp4'
os.environ['AVATAR_MAX_SIDE']  = '512'   # T4-friendly
os.environ['AVATAR_FP16']      = '1'
os.environ['AVATAR_FPS']       = '25'

In [ ]:
# 5) Jalankan server + tunnel publik. Salin URL https://xxxx.trycloudflare.com yang muncul.
import sys
sys.path.insert(0, '/content/yt-short/colab')
from flask import Flask
from flask_cloudflared import run_with_cloudflared
from avatar_server import register_avatar_routes

app = Flask(__name__)
register_avatar_routes(app)
run_with_cloudflared(app)   # cetak URL publik
app.run(port=7860)